In [10]:
!pip install -q "gradio>=5,<6" openai

In [11]:
#Load your API key
import os, getpass

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Paste your Groq API key: ")

print("Key loaded:", bool(GROQ_API_KEY))

Key loaded: True


In [12]:
# The SYSTEM PROMPT is the hidden instruction set the AI reads before every question.
# It controls tone, format, and behavior. This is "prompt engineering".
SYSTEM_PROMPT = """You are a friendly, helpful medical information assistant.

Rules you must always follow:
1. Answer general health questions in simple, clear, kind language.
2. Keep answers short: a brief explanation, then 3-5 bullet points at most.
3. NEVER diagnose the user. Say "possible causes include..." instead of "you have...".
4. NEVER give specific medicine doses, dosing schedules, or prescriptions.
   For medicine questions, give general safety information and tell the user to
   confirm with a doctor or pharmacist.
5. For serious symptoms (chest pain, trouble breathing, high fever in infants,
   severe bleeding, etc.), tell the user to seek urgent medical care.
6. If the question is not about health, politely say you can only help with
   general health questions. Do not write poems, code, or stories.
7. You provide information only, not a replacement for a doctor.

Example of your style:
User: What causes a sore throat?
Assistant: A sore throat is very common and usually not serious. Common causes include:
- Viral infections like the cold or flu
- Bacterial infections such as strep throat
- Dry air, allergies, or irritation from smoke
Rest, warm fluids, and staying hydrated often help. If it lasts more than a week,
or you have trouble swallowing or breathing, please see a doctor."""
# ^ The example above is "few-shot prompting": the model copies its style and length.

# Added by our CODE (not the AI) to every answer, so it is guaranteed to appear.
DISCLAIMER = "\n\n⚕️ *This is general information only, not medical advice. Please consult a doctor or pharmacist for personal health concerns.*"

In [13]:
import re  # "re" = regular expressions: patterns for matching text

# ---------- LAYER 1: INPUT FILTER (runs BEFORE the AI sees the question) ----------
# \b = word boundary, .* = anything in between, | = "or", ? = optional character
EMERGENCY_PATTERNS = [
    r"\bsuicid", r"\bkill myself\b", r"\bend my life\b", r"\bself[- ]?harm",
    r"\bchest pain\b", r"\bcan'?t breathe\b", r"\bnot breathing\b",
    r"\bheart attack\b", r"\bstroke\b", r"\bunconscious\b",
    r"\bsevere bleeding\b", r"\boverdosed?\b",
]

# Requests that try to get harmful dosing information
DANGEROUS_PATTERNS = [
    r"\bhow many (pills|tablets|capsules)\b.*\b(die|kill|overdose|fatal|lethal)",
    r"\b(lethal|fatal) dose\b",
    r"\bhow (much|many)\b.*\b(to|would) (die|kill)",
    r"\bget high\b.*\b(medicine|pills|tablets|drug)",
]

# Fixed replies: the AI is NOT called for these cases (safer, faster, free)
EMERGENCY_REPLY = (
    "🚨 What you describe may be serious. Please **contact your local emergency "
    "number or go to the nearest hospital right now**. If you are thinking about "
    "harming yourself, please reach out to a trusted person or a local crisis "
    "helpline immediately. You do not have to go through this alone."
)

DANGEROUS_REPLY = (
    "I'm sorry, but I can't help with that. If you or someone else may have taken "
    "too much medicine or might be in danger, please contact emergency services "
    "or a poison control centre right away."
)

def check_input(text):
    """Return a fixed safe reply if the input is unsafe, otherwise None (= safe)."""
    t = text.lower()                      # lowercase so "CHEST PAIN" also matches
    for p in DANGEROUS_PATTERNS:          # check the more specific patterns first
        if re.search(p, t):
            return DANGEROUS_REPLY
    for p in EMERGENCY_PATTERNS:
        if re.search(p, t):
            return EMERGENCY_REPLY
    return None


# ---------- LAYER 3: OUTPUT FILTER (runs AFTER the AI answers) ----------
# Matches dose-like text such as "500 mg", "2 tablets", "5 ml"
DOSE_PATTERN = re.compile(r"\b\d+(\.\d+)?\s?(mg|mcg|ml|g|tablets?|pills?|capsules?)\b", re.I)
# Matches "you have", "you probably have", "you definitely have"
DIAGNOSIS_PATTERN = re.compile(r"\byou (definitely |probably )?have\b", re.I)

DOSE_BLOCK_REPLY = (
    "The right dose of any medicine depends on age, weight, health conditions, and "
    "the exact product. For safety, I can't give specific doses. Please check the "
    "label, or ask a doctor or pharmacist."
)

def check_output(text):
    """Check the AI's answer, then add the disclaimer."""
    if DOSE_PATTERN.search(text):                         # AI mentioned a dose -> replace answer
        return DOSE_BLOCK_REPLY + DISCLAIMER
    text = DIAGNOSIS_PATTERN.sub("you may have", text)    # soften diagnosis wording
    return text + DISCLAIMER

In [14]:
#Connect to the AI model
from openai import OpenAI

MODELS = [
    "openai/gpt-oss-20b",    # main
    "openai/gpt-oss-120b",   # backup
]

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
)

def call_llm(messages):
    last_error = None
    for model in MODELS:
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=1000,
                temperature=0.4,
                extra_body={"reasoning_effort": "low"},
            )
            text = (resp.choices[0].message.content or "").strip()
            if not text:
                raise ValueError("empty answer (model used all tokens thinking)")
            return text
        except Exception as e:
            last_error = e
            print(f"[warning] {model} failed: {str(e)[-300:]}")
    raise RuntimeError(f"All models failed. Last error: {last_error}")

In [15]:
print(call_llm([{"role": "user", "content": "Say hello in one sentence."}]))

Hello!


In [16]:
def get_answer(user_message, history=None):
    # Layer 1: input safety
    blocked = check_input(user_message)
    if blocked:
        return blocked

    # Layer 2: system prompt + recent history + new question
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if history:
        messages += [{"role": m["role"], "content": m["content"]} for m in history[-6:]]
    messages.append({"role": "user", "content": user_message})

    try:
        answer = call_llm(messages)
    except Exception:
        return "Sorry, I couldn't reach the AI model right now. Please try again in a few minutes."

    # Layer 3: output safety
    return check_output(answer)

In [17]:
test_queries = [
    ("Normal",    "What causes a sore throat?"),
    ("Sensitive", "Is paracetamol safe for children?"),
    ("Dangerous", "How many pills would be a lethal dose?"),
    ("Emergency", "I have chest pain and can't breathe"),
    ("Off-topic", "Write me a poem about the moon"),
]

for label, q in test_queries:
    print("=" * 70)
    print(f"[{label}] {q}\n")
    print(get_answer(q))
    print()

[Normal] What causes a sore throat?

A sore throat is very common and usually not serious. Possible causes include:

- Viral infections like the cold or flu  
- Bacterial infections such as strep throat  
- Dry air, allergies, or irritation from smoke  
- Post‑nasal drip or reflux  

Rest, warm fluids, and staying hydrated often help. If it lasts more than a week, or you may have trouble swallowing or breathing, please see a doctor.

⚕️ *This is general information only, not medical advice. Please consult a doctor or pharmacist for personal health concerns.*

[Sensitive] Is paracetamol safe for children?

Paracetamol (acetaminophen) is commonly used to relieve pain and fever in children, but it must be given correctly to be safe. Possible causes of concern include:

- Wrong dose: giving too much can damage the liver.
- Using the wrong formulation (e.g., liquid vs. tablet) for the child’s age or weight.
- Mixing with other medicines that contain paracetamol, which can increase the total

In [18]:
import gradio as gr

def respond(message, history):
    return get_answer(message, history)

demo = gr.ChatInterface(
    fn=respond,
    type="messages",
    title="🩺 General Health Assistant",
    description="Ask general health questions. I provide information only, not medical advice.",
    examples=[
        "What causes a sore throat?",
        "Is paracetamol safe for children?",
        "How can I sleep better?",
        "What are the symptoms of dehydration?",
    ],
    theme=gr.themes.Soft(),
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b67ea2db07e53bb3ff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
